In [3]:
## Cargue de la información
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [14]:
bodegas = pd.read_csv('bodegas_anon.csv', sep=';', encoding='utf-8', header=None)
nombres_columnas_bodegas = ['id_bodega', 'ciudad', 'nombre_bodega', 'tipo', 'region', 'pais', 'antiguedad_dias']
bodegas.columns = nombres_columnas_bodegas

In [15]:
bodegas.head()

,id_bodega,ciudad,nombre_bodega,tipo,region,pais,fecha
0,STORE_000000,CITY_000006,1 - Administrativas,NaN,STORE_DESC_000266,ÃLITE,80.0
1,STORE_000001,CITY_000006,1 - Comerciales (No tiendas),STORE_TYPE_000010,STORE_DESC_000114,URBAN,NaN
2,STORE_000002,CITY_000006,1 - Produccion,NaN,STORE_DESC_000310,URBAN,NaN
3,STORE_000003,NaN,1 - CEDI Baguer,NaN,STORE_DESC_000268,URBAN,NaN
4,STORE_000004,NaN,1 - Administrativas,NaN,STORE_DESC_000009,URBAN,NaN


In [19]:
bodegas.describe()

,fecha
count,147.000000
mean,132.918367
std,92.282585
min,0.000000
25%,72.000000
50%,121.000000
75%,171.000000
max,607.000000


In [ ]:
eventos = pd.read_csv('eventos_anon.csv', sep=';', encoding='utf-8', header=None)
nombres_columnas_eventos = ['fecha', 'id_producto', 'id_bodega', 'tipo_evento', 'cantidad', 'cantidad_2', 'precio', 'precio_total', 'costo', 'col_extra_1', 'col_extra_2', 'col_extra_3', 'id_ticket']
eventos.columns = nombres_columnas_eventos
eventos.head()

,0,1,2,3,4,5,6,7,8,9,10,11,12
0,2024-10-05 00:00:00.000,PROD_000000,STORE_000048,venta,1,1.0,923950.0,692962.5,325637.0,25,0,0,G120-4998
1,2024-12-21 00:00:00.000,PROD_000000,STORE_000146,venta,1,1.0,970950.0,825307.5,325637.0,15,0,0,G440-13393
2,2025-03-17 00:00:00.000,PROD_000000,STORE_000225,venta,1,1.0,970950.0,922402.5,448758.0,0,5,0,G670-19863
3,2024-09-14 00:00:00.000,PROD_000000,STORE_000312,venta,1,1.0,923950.0,739160.0,325637.0,20,0,0,G829-4846
4,2025-07-12 00:00:00.000,PROD_000000,STORE_000060,venta,1,1.0,970950.0,970950.0,448758.0,0,0,0,X240-1217


In [17]:
productos = pd.read_csv('productos_anon.csv', sep=';', encoding='utf-8', header=None)
nombres_columnas_productos = ['id_producto', 'desc_producto', 'talla', 'color', 'marca', 'genero', 'categoria', 'subcategoria', 'patron', 'largo', 'temporada', 'tipo', 'material']
productos.columns = nombres_columnas_productos
productos.head()

,0,1,2,3,4,5,6,7,8,9,10,11,12
0,PROD_033272,PROD_DESC_007656,12,BL,BRAND_000021,FEMENINO,PANTALON,Recta,UNICOLOR,LARGO,2025-3 (COMPRA Oct 01 - Dic 31),TYPE_000000,TEXTIL
1,PROD_033273,PROD_DESC_007656,04,NG,BRAND_000021,FEMENINO,PANTALON,Recta,UNICOLOR,LARGO,2025-3 (COMPRA Oct 01 - Dic 31),TYPE_000000,TEXTIL
2,PROD_033262,PROD_DESC_002831,14,CF,BRAND_000021,FEMENINO,PANTALON,Cargo,UNICOLOR,LARGO,2025-2 (COMPRA Jul 01 - Sep 30),TYPE_000000,TEXTIL
3,PROD_033261,PROD_DESC_002831,12,CF,BRAND_000021,FEMENINO,PANTALON,Cargo,UNICOLOR,LARGO,2025-2 (COMPRA Jul 01 - Sep 30),TYPE_000000,TEXTIL
4,PROD_020811,PROD_DESC_009825,LL,BG,BRAND_000021,FEMENINO,CONJUNTO,PANTALON,UNICOLOR,LARGO,2025-3 (COMPRA Oct 01 - Dic 31),TYPE_000000,TEXTIL


In [18]:
stock = pd.read_csv('stock_anon.csv', sep=';', encoding='utf-8', header=None)
nombres_columnas_stock = ['id_producto', 'id_bodega', 'cantidad', 'fecha_corte']
stock.columns = nombres_columnas_stock
stock.head()

,0,1,2,3
0,PROD_012861,STORE_000000,0.0,2026-03-30
1,PROD_034500,STORE_000000,0.0,2026-03-30
2,PROD_026408,STORE_000004,-2.0,2026-03-30
3,PROD_019068,STORE_000004,1.0,2026-03-30
4,PROD_019072,STORE_000004,1.0,2026-03-30


## Análisis de Calidad de Datos
A continuación se revisan los valores nulos e inconsistencias iniciales.

In [ ]:
# Valores nulos en bodegas
print('Nulos en bodegas:')
print(bodegas.isnull().sum())

In [ ]:
# Revisión de inventarios negativos en stock
stock_negativo = stock[stock['cantidad'] < 0]
print(f'Registros con stock negativo: {len(stock_negativo)}')
display(stock_negativo.head())

## Preparación de Datos: Modelo de Elasticidad de Precios
Para un modelo de precios dinámicos, la granularidad ideal es analizar la relación entre el **Precio** y la **Demanda (Cantidad)** a nivel de `[Fecha, Producto, Bodega]`.

Los pasos para unificar las tablas son:
1. **Agrupar transacciones (eventos):** Filtrar solo las ventas y calcular la cantidad total vendida y el precio unitario promedio ponderado por día.
2. **Cruzar dimensiones:** Unir características del producto (`productos`) y de la ubicación geográfica/tipo de tienda (`bodegas`).
3. **Cruzar inventario (`stock`):** Añadir el stock disponible. *Nota analítica: Dado que el stock proporcionado es solo una foto actual (2026-03-30), servirá como referencia final. Para evitar sesgos en el modelo real de elasticidad, lo ideal sería contar con el stock histórico diario para identificar 'quiebres de stock' (out-of-stock), es decir, días donde la venta fue cero porque no había producto, y no porque el precio fuera muy alto.*

In [ ]:
# 1. Filtrar solo las ventas dentro de los eventos
ventas = eventos[eventos['tipo_evento'] == 'venta'].copy()

# 2. Agrupar a nivel de Fecha, Producto y Bodega
ventas_diarias = ventas.groupby(['fecha', 'id_producto', 'id_bodega']).agg(
    cantidad_vendida=('cantidad', 'sum'),
    ingreso_total=('precio_total', 'sum')
).reset_index()

# Calcular el precio real promedio unitario de ese día
ventas_diarias['precio_promedio'] = ventas_diarias['ingreso_total'] / ventas_diarias['cantidad_vendida']

# 3. Unir (Merge) con la tabla de Productos (Left join)
df_unificado = ventas_diarias.merge(productos, on='id_producto', how='left')

# 4. Unir (Merge) con la tabla de Bodegas (Left join)
df_unificado = df_unificado.merge(bodegas, on='id_bodega', how='left')

# 5. Unir (Merge) con la foto de Stock (Left join)
# Extraemos solo las columnas necesarias y renombramos 'cantidad' para no generar sufijos _x o _y
stock_clean = stock[['id_producto', 'id_bodega', 'cantidad']].rename(columns={'cantidad': 'stock_actual'})
df_unificado = df_unificado.merge(stock_clean, on=['id_producto', 'id_bodega'], how='left')

# Rellenar con 0 donde no tengamos registro de stock en la foto
df_unificado['stock_actual'] = df_unificado['stock_actual'].fillna(0)

# Mostrar el resultado de la Tabla Analítica Unificada (Master Table)
df_unificado.head()

## Exportar la Tabla Maestra para GitHub
Para subir este archivo a GitHub, el formato más recomendado es **Parquet** (`.parquet`). 

**¿Por qué Parquet?**
- **Compresión extrema:** Un archivo de 500MB en CSV puede pesar solo 50MB en Parquet, lo cual es vital para no superar el límite de **100MB por archivo** de GitHub.
- **Conserva los tipos de datos:** Guarda nativamente qué columna es fecha (`datetime`), cuál es texto y cuál es número. No tendrás que volver a parsear fechas al cargarlo.
- **Velocidad:** Es muchísimo más rápido de leer y escribir que un CSV.

*(Como alternativa secundaria, también se incluye el código para exportar a un CSV comprimido con GZIP)*.

In [ ]:
# 1. Exportar en formato Parquet (El campeón indiscutible para GitHub y Data Science)
# Nota: Si te marca error, asegúrate de tener la librería instalada corriendo: !pip install pyarrow
try:
    df_unificado.to_parquet('dataset_elasticidad.parquet', index=False)
    print("✅ Archivo Parquet exportado con éxito: 'dataset_elasticidad.parquet'")
except Exception as e:
    print("⚠️ Error con Parquet (posiblemente falte pyarrow). Error:", e)

# 2. Exportar a CSV Comprimido (.gz) (Por si necesitas abrirlo en herramientas que no soporten Parquet)
df_unificado.to_csv('dataset_elasticidad.csv.gz', compression='gzip', index=False)
print("✅ Archivo CSV comprimido exportado con éxito: 'dataset_elasticidad.csv.gz'")

"""
💡 ¿Cómo cargar estos archivos mañana o en otro proyecto?
Simplemente usas:

import pandas as pd
df_modelo = pd.read_parquet('dataset_elasticidad.parquet')
"""

## Optimización: Exportar tablas base a Parquet
Para no tener que volver a leer los archivos `.csv` pesados en el futuro (especialmente `eventos_anon.csv` que tiene 8 millones de filas), aquí guardamos las tablas base en formato **Parquet**.

In [ ]:
# 1. Exportar las 4 tablas originales a Parquet
print("Exportando tablas a Parquet... esto tomará unos segundos.")

eventos.to_parquet('eventos.parquet', index=False)
productos.to_parquet('productos.parquet', index=False)
bodegas.to_parquet('bodegas.parquet', index=False)
stock.to_parquet('stock.parquet', index=False)

print("✅ Todas las tablas han sido exportadas a Parquet exitosamente.")

### Cargar los datos desde Parquet (Para futuras sesiones)
La próxima vez que abras esta libreta, en lugar de usar `pd.read_csv(...)`, puedes simplemente ejecutar la celda de abajo para cargar todo en una fracción de segundo.

In [ ]:
# 2. Código rápido para cargar todo tu entorno de trabajo en segundos
import pandas as pd

# Cargar desde los archivos Parquet optimizados
eventos = pd.read_parquet('eventos.parquet')
productos = pd.read_parquet('productos.parquet')
bodegas = pd.read_parquet('bodegas.parquet')
stock = pd.read_parquet('stock.parquet')

print("✅ Datos cargados a máxima velocidad.")
print(f"Filas en eventos: {len(eventos):,}")